In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
import torch.nn as nn
import pandas as pd
import numpy as np
from hasoc_model import encode_labels
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [ ]:
def encode_labels(df):
    df = df.copy()
    df["label_A_enc"] = df["label_A"].map({"NOT": 0, "HOF": 1})
    df["label_B_enc"] = df["label_B"].map({"HATE": 0, "OFFN": 1, "PRFN": 2})
    df["label_C_enc"] = df["label_C"].map({"UNT": 0, "TIN": 1}) 
    return df.dropna(subset=["label_A_enc"])

In [2]:
class Paola(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased", num_outputs=8, bin_outputs=5):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_outputs)
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, bin_outputs),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.regressor(pooled), self.classifier(pooled)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_paola = Paola()

# Load weights BEFORE wrapping in DataParallel
state_dict = torch.load("../berkeley_model/best_Berkeley_model.pth", map_location=device, weights_only=True)
model_paola.load_state_dict(state_dict)

# Then wrap in DataParallel if multiple GPUs
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
    model_paola = nn.DataParallel(model_paola)

model_paola = model_paola.to(device)
print("best_Berkeley_model.pth loaded and ready to use!")

tokenizer_paola = AutoTokenizer.from_pretrained("distilbert-base-uncased")

model2_loaded.pth loaded and ready to use!


In [4]:
new_feature_names = ['sentiment', 'respect', 'insult', 'humiliate', 'status',
                  'dehumanize', 'attack_defend', 'hatespeech',
                     'target_race', 'target_religion', 'target_origin', 'target_gender',
                'target_sexuality']

In [5]:
# -----TRAIN-----

In [ ]:
df_clara_train = pd.read_csv("../hasoc_model/hasoc_dataset/train.tsv", sep="\t")
df_clara_train.columns = ["id", "text", "label_A", "label_B", "label_C"]
df_clara_train = df_clara_train[["text", "label_A", "label_B", "label_C"]] 
df_clara_train = encode_labels(df_clara_train)
#df_clara_train = df_clara_train[0:200]
print(df_clara_train.head())

                                                text label_A label_B label_C  \
0  #DhoniKeepsTheGlove | WATCH: Sports Minister K...     NOT    NONE    NONE   
1  @politico No. We should remember very clearly ...     HOF    HATE     TIN   
2  @cricketworldcup Guess who would be the winner...     NOT    NONE    NONE   
3  Corbyn is too politically intellectual for #Bo...     NOT    NONE    NONE   
4  All the best to #TeamIndia for another swimmin...     NOT    NONE    NONE   

   label_A_enc  label_B_enc  label_C_enc  
0            0          NaN          NaN  
1            1          0.0          1.0  
2            0          NaN          NaN  
3            0          NaN          NaN  
4            0          NaN          NaN  


In [7]:
encodings_paola_train = tokenizer_paola(df_clara_train["text"].tolist(), truncation=True, padding=True, max_length=128, return_tensors="pt")

model_paola.eval()
input_ids_paola_train = encodings_paola_train['input_ids'].to(device)
attention_mask_paola_train = encodings_paola_train['attention_mask'].to(device)

with torch.no_grad():
    preds_num_train, preds_bin_train = model_paola(input_ids=input_ids_paola_train, attention_mask=attention_mask_paola_train)

preds_num_train = preds_num_train.cpu().numpy()
preds_bin_train = preds_bin_train.cpu().numpy()
preds_bin_train = (preds_bin_train > 0.5).astype(int)

for idx in range (3):
    print(f"Sentence: {df_clara_train["text"].tolist()[idx]}")
    print(f"Numerical predictions: {preds_num_train[idx]}")
    print(f"Binary predictions: {preds_bin_train[idx]}")
    print()


Sentence: #DhoniKeepsTheGlove | WATCH: Sports Minister Kiren Rijiju issues statement backing MS Dhoni over 'Balidaan Badge', tells BCCI to take up the matter with ICC and keep government in the know as nation's pride is involved    https://t.co/zuo5335Rjr
Numerical predictions: [1.4261338  1.026587   0.7156261  0.6578899  1.8180861  0.52794576
 1.3764595  0.03854384]
Binary predictions: [0 0 0 0 1]

Sentence: @politico No. We should remember very clearly that #Individual1 just admitted to treason . #TrumpIsATraitor  #McCainsAHero #JohnMcCainDay
Numerical predictions: [2.8258023  2.305354   1.895153   1.7395172  2.1474311  1.330773
 2.071613   0.32836255]
Binary predictions: [0 0 0 0 0]

Sentence: @cricketworldcup Guess who would be the winner of this #CWC19?     Team who gets maximum points from the abandoned matches 😄 #ShameOnICC #WIvsENG @ICC
Numerical predictions: [2.2002656  1.9434769  1.5539635  1.3744226  2.0864556  0.9715349
 1.9307483  0.07670166]
Binary predictions: [1 0 0 0 0

In [8]:
combined_preds_train = np.concatenate([preds_num_train, preds_bin_train], axis=1)
preds_df_train = pd.DataFrame(combined_preds_train, columns=new_feature_names)
df_clara_train = pd.concat([df_clara_train.reset_index(drop=True), preds_df_train], axis=1)
print(df_clara_train.head())
df_clara_train.to_csv("../hasoc_model/hasoc_dataset/hasoc_dataset_with_features_train.tsv", sep='\t', index=False)

                                                text label_A label_B label_C  \
0  #DhoniKeepsTheGlove | WATCH: Sports Minister K...     NOT    NONE    NONE   
1  @politico No. We should remember very clearly ...     HOF    HATE     TIN   
2  @cricketworldcup Guess who would be the winner...     NOT    NONE    NONE   
3  Corbyn is too politically intellectual for #Bo...     NOT    NONE    NONE   
4  All the best to #TeamIndia for another swimmin...     NOT    NONE    NONE   

   label_A_enc  label_B_enc  label_C_enc  sentiment   respect    insult  \
0            0          NaN          NaN   1.426134  1.026587  0.715626   
1            1          0.0          1.0   2.825802  2.305354  1.895153   
2            0          NaN          NaN   2.200266  1.943477  1.553964   
3            0          NaN          NaN   2.947032  2.803069  2.523936   
4            0          NaN          NaN   0.699087  0.675581  0.431476   

   humiliate    status  dehumanize  attack_defend  hatespeech  targe

In [9]:
# -----TEST-----

In [ ]:
df_clara_test = pd.read_csv("../hasoc_model/hasoc_dataset/test.tsv", sep="\t")
df_clara_test.columns = ["id", "text", "label_A", "label_B", "label_C"]
df_clara_test = df_clara_test[["text", "label_A", "label_B", "label_C"]] 
df_clara_test = encode_labels(df_clara_test)
#df_clara_test = df_clara_test[0:200]
print(df_clara_test.head())

                                                text label_A label_B label_C  \
0  West Bengal Doctor Crisis: Protesting doctors ...     NOT    NONE    NONE   
1  68.5 million people have been forced to leave ...     NOT    NONE    NONE   
2  You came, you saw .... we will look after the ...     NOT    NONE    NONE   
3  We'll get Brexit delivered by October 31st.   ...     NOT    NONE    NONE   
4  Fuck you. Go back to the dark ages you cow @IB...     HOF    PRFN     UNT   

   label_A_enc  label_B_enc  label_C_enc  
0            0          NaN          NaN  
1            0          NaN          NaN  
2            0          NaN          NaN  
3            0          NaN          NaN  
4            1          2.0          0.0  


In [14]:
encodings_paola_test = tokenizer_paola(df_clara_test["text"].tolist(), truncation=True, padding=True, max_length=128, return_tensors="pt")

model_paola.eval()
input_ids_paola_test = encodings_paola_test['input_ids'].to(device)
attention_mask_paola_test = encodings_paola_test['attention_mask'].to(device)

with torch.no_grad():
    preds_num_test, preds_bin_test = model_paola(input_ids=input_ids_paola_test, attention_mask=attention_mask_paola_test)

preds_num_test = preds_num_test.cpu().numpy()
preds_bin_test = preds_bin_test.cpu().numpy()
preds_bin_test = (preds_bin_test > 0.5).astype(int)

for idx in range (3):
    print(f"Sentence: {df_clara_test["text"].tolist()[idx]}")
    print(f"Numerical predictions: {preds_num_test[idx]}")
    print(f"Binary predictions: {preds_bin_test[idx]}")
    print()


Sentence: West Bengal Doctor Crisis: Protesting doctors agree to meet Mamata Banerjee in presence of full media even as IMA goes on strike  
Numerical predictions: [ 2.0381958   1.6860917   1.2013574   1.0330188   2.054336    0.6491018
  1.9303454  -0.11902817]
Binary predictions: [0 0 1 0 0]

Sentence: 68.5 million people have been forced to leave their homes.      Read more: https://wef.ch/2YQcwpk  #refugees #society
Numerical predictions: [ 1.7563808   1.0353525   0.58938676  0.5233234   2.0910969   0.57387435
  0.98687696 -0.02522886]
Binary predictions: [0 0 1 0 0]

Sentence: You came, you saw .... we will look after the fort! Good luck! 
Numerical predictions: [1.3560685 1.45803   1.1453875 1.0515575 1.9170464 0.8216615 1.6690835
 0.1749136]
Binary predictions: [0 0 1 0 0]



In [15]:
combined_preds_test = np.concatenate([preds_num_test, preds_bin_test], axis=1)
preds_df_test = pd.DataFrame(combined_preds_test, columns=new_feature_names)
df_clara_test = pd.concat([df_clara_test.reset_index(drop=True), preds_df_test], axis=1)

df_clara_test.to_csv("../hasoc_model/hasoc_dataset/hasoc_dataset_with_features_test.tsv", sep='\t', index=False)